# Lab 3.2 — Training a Graph Convolutional Network
**Module III · Graph Neural Networks**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DanielFPerez/llm-gnns-course_labs/blob/main/module-3-gnn/lab3_2_training_gcn.ipynb)

---

## What you will do
1. Train an **MLP baseline** on Cora node features — ignoring the graph structure.
2. Train a **2-layer GCN** (Kipf & Welling, 2017) that aggregates neighbour features.
3. Compare the two models and quantify how much the **graph structure** contributes.
4. Inspect **node embeddings** with a t-SNE projection to see how GCNs cluster papers by topic.
5. `[Extension]` Experiment with depth: does a deeper GCN help?

## Prerequisites
Lab 3.1 completed. Basic PyTorch familiarity.

**Estimated time:** 55–70 min

---
## 0 · Setup

In [ ]:
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/DanielFPerez/llm-gnns-course_labs.git",
         "labs_assignment"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         "labs_assignment/environment/requirements.txt"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "torch-geometric"], check=True)
    sys.path.insert(0, "labs_assignment")
else:
    sys.path.insert(0, str(Path("..").resolve()))

print("Setup complete.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

from torch_geometric.datasets import Planetoid
import torch_geometric.transforms as T
from torch_geometric.nn import GCNConv

from utils import plot_training_curves, plot_embeddings, check_gnn_model

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {DEVICE}")

In [ ]:
dataset = Planetoid(root="/tmp/Cora", name="Cora",
                    transform=T.NormalizeFeatures())
data = dataset[0].to(DEVICE)

CLASS_NAMES = [
    "Case-Based", "Genetic Algorithms", "Neural Networks",
    "Probabilistic Methods", "Reinforcement Learning",
    "Rule Learning", "Theory",
]

print(data)

---
## 1 · MLP baseline — ignoring the graph

Before bringing in the graph, we establish a **baseline**: a plain multi-layer perceptron that classifies each node using only its own feature vector — as if the graph edges did not exist. This is exactly what a standard classifier (like the Logistic Regression in Module I) would do.

The MLP sees each node independently:
$$h_i = \text{ReLU}(W_1 \, x_i) \quad \text{then} \quad \hat{y}_i = W_2 \, h_i$$

**Why does this matter?** The gap between MLP and GCN is the direct, measurable contribution of the graph structure to prediction quality. If they perform the same, the edges contain no useful information. If GCN outperforms MLP substantially, the neighbourhood context is critical.

### Exercise 3.2.1 `[Core]` — Build the MLP

Implement `MLP` as a `nn.Module` with:
- A first linear layer: `in_features → hidden_channels`
- ReLU activation + dropout
- A second linear layer: `hidden_channels → out_channels`

The `forward` method receives `x` (node features) and ignores `edge_index`.

In [ ]:
# YOUR CODE HERE
# Hint: Implement MLP as a nn.Module with:

class MLP(nn.Module):
    pass  # replace this

### Training loop helper

Both the MLP and the GCN use the same training loop. We define it once here — this is a good practice that makes the comparison fair.

In [ ]:
def train_epoch(model, data, optimiser):
    model.train()
    optimiser.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimiser.step()
    return loss.item()


@torch.no_grad()
def evaluate(model, data):
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    results = {}
    for split in ["train", "val", "test"]:
        mask = getattr(data, f"{split}_mask")
        correct = (pred[mask] == data.y[mask]).sum().item()
        results[split] = correct / mask.sum().item()
    return results


def train_model(model, data, n_epochs=200, lr=0.01, weight_decay=5e-4):
    optimiser = torch.optim.Adam(model.parameters(), lr=lr,
                                 weight_decay=weight_decay)
    history = {"train_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(model, data, optimiser)
        metrics = evaluate(model, data)
        history["train_loss"].append(loss)
        history["train_acc"].append(metrics["train"])
        history["val_acc"].append(metrics["val"])

        if epoch % 50 == 0:
            print(f"Epoch {epoch:>3} | loss {loss:.4f} | "
                  f"train acc {metrics['train']:.3f} | val acc {metrics['val']:.3f}")

    test_metrics = evaluate(model, data)
    print(f"\n→ Final TEST accuracy: {test_metrics['test']:.3f}")
    return history, test_metrics

### Exercise 3.2.2 `[Core]` — Train the MLP

Train the MLP for 200 epochs with `lr=0.01` and `weight_decay=5e-4`. Then:
1. Plot the training curves with `plot_training_curves(history)`.
2. Note the final test accuracy. Expect **~55–65%**.
3. Run `check_gnn_model("3.2.2", mlp, data, split="test", min_acc=0.50)`.

In [ ]:
# YOUR CODE HERE
# Hint: Train the MLP for 200 epochs with lr=0.01 and weight_decay=5e-4. Then:
raise NotImplementedError("Complete this exercise")

> **Baseline result:** The MLP achieves ~55–65% test accuracy using only node features (bag-of-words). Not bad for just 140 training examples, but there is clearly room to improve — and we have not used the graph at all yet.

---
## 2 · Graph Convolutional Network

The **GCN** (Kipf & Welling, ICLR 2017) adds graph structure via **message passing**: each node aggregates the feature vectors of its neighbours before applying a linear transformation.

The GCN propagation rule for layer $l$:
$$H^{(l+1)} = \sigma\!\left( \tilde{D}^{-1/2} \tilde{A} \tilde{D}^{-1/2} \, H^{(l)} W^{(l)} \right)$$

Where $\tilde{A} = A + I$ (adjacency with self-loops), $\tilde{D}$ is its degree matrix. In practice this is:
1. Add a self-loop to each node (include its own features).
2. Normalise by $1 / \sqrt{\deg(u) \cdot \deg(v)}$ for each edge $(u, v)$.
3. Sum normalised neighbour features.
4. Apply a learnable linear transformation $W$ and an activation.

PyG's `GCNConv` implements all of this in one line:
```python
from torch_geometric.nn import GCNConv
self.conv1 = GCNConv(in_channels, hidden_channels)
```

The call signature is:
```python
x = self.conv1(x, edge_index)   # <-- note the edge_index argument
```

### Exercise 3.2.3 `[Core]` — Build the GCN

Implement `GCN` as a `nn.Module` with:
- A first `GCNConv` layer: `in_channels → hidden_channels`
- ReLU activation + dropout
- A second `GCNConv` layer: `hidden_channels → out_channels`

The `forward(x, edge_index)` method passes `edge_index` to both conv layers.

In [ ]:
# YOUR CODE HERE
# Hint: Implement GCN as a nn.Module with:

class GCN(nn.Module):
    pass  # replace this

### Exercise 3.2.4 `[Core]` — Train the GCN and compare

Train the GCN for 200 epochs with the same hyperparameters as the MLP. Then:
1. Plot the training curves.
2. Print and compare the test accuracies of MLP vs GCN.
3. Run `check_gnn_model("3.2.4", gcn, data, split="test", min_acc=0.78)`.

In [ ]:
# YOUR CODE HERE
# Hint: Train the GCN for 200 epochs with the same hyperparameters as the MLP. Then:
raise NotImplementedError("Complete this exercise")

> **The key result:** The GCN achieves ~80–82% test accuracy versus ~55–65% for the MLP — a **15–25 percentage point improvement** from incorporating the graph. With only 140 training examples (20 per class!), the graph structure provides crucial regularisation: nodes tend to have the same label as their neighbours, so a GCN learns to exploit this local homophily.
>
> This is a clear, quantitative demonstration of why GNNs exist. When relationships between entities carry signal — citations, friendships, transactions — ignoring those relationships leaves significant accuracy on the table.

---
## 3 · Visualising what the GCN learned

The intermediate representation after the first GCN layer is a **node embedding** — a learned, lower-dimensional representation of each node that incorporates its neighbourhood. We can project these to 2D with t-SNE to see whether the GCN has learned to cluster papers by topic.

### Exercise 3.2.5 `[Core]` — Extract and visualise GCN embeddings

Extract the **intermediate node embeddings** from the trained GCN (the output of the first conv layer + ReLU, before the second conv layer). Project them to 2D with t-SNE and colour by class.

Then do the same for the raw input features `data.x` and the **MLP** intermediate activations. Compare all three:
- Do the GCN embeddings form tighter clusters?

**Hint:** Add a helper method to GCN that returns embeddings:

In [ ]:
# YOUR CODE HERE
# Hint: Extract the intermediate node embeddings from the trained GCN (the output of the first conv layer + ReLU, before the ...
raise NotImplementedError("Complete this exercise")

> **Key observation:** The GCN embeddings form much tighter, more separated clusters than either the raw features or the MLP hidden states. This is because the GCN mixes each node's features with its neighbours' features, effectively denoising and smoothing within classes. Papers that share many co-cited references are pulled closer together in embedding space, even if their individual bag-of-words are slightly different.

---
## 4 · `[Extension]` How deep should the GCN be?

A $k$-layer GCN aggregates information from the $k$-hop neighbourhood of each node. Naively, deeper should be better — more context. But in practice, very deep GCNs suffer from **over-smoothing**: with enough layers, all node representations converge to the same vector, losing discriminative information.

### Exercise 3.2.6 `[Extension]` — Depth ablation

Implement a `DeepGCN` class with a configurable number of layers. Train versions with 1, 2, 3, 4, and 6 layers, recording the final test accuracy for each. Plot test accuracy vs. number of layers.

At what depth does performance peak? When does over-smoothing set in?

In [ ]:
# YOUR CODE HERE
# Hint: Implement a DeepGCN class with a configurable number of layers. Train versions with 1, 2, 3, 4, and 6 layers, recordi...

class DeepGCN(nn.Module):
    pass  # replace this

> **Key observation:** Performance peaks at **2 layers** (~81%) and degrades sharply beyond 3–4 layers. This is **over-smoothing**: with each additional GCN layer, node representations are further averaged with their neighbours. By depth 6, most representations have converged to nearly the same vector — the model can no longer distinguish nodes.
>
> This is one of the fundamental limitations of message-passing GNNs — and a core motivation for **Graph Transformers** (Module IV), which use global attention instead of local aggregation and therefore do not suffer from the same depth problem.

---
## Summary

| Model | Test accuracy | Key property |
|---|---|---|
| MLP (no graph) | ~60% | Feature-only baseline |
| 2-layer GCN | ~81% | +~20 pp from 1-hop neighbourhood aggregation |
| 3+ layer GCN | Degrades | Over-smoothing |

**Takeaways:**
- The graph structure adds ~20 percentage points on Cora — a dramatic improvement from very few labelled examples.
- GCN is a symmetric, degree-normalised message-passing scheme: all neighbours contribute equally.
- Over-smoothing limits depth; shallow (2-layer) GCNs are often best.
- Embedding visualisation confirms the GCN learns class-separated representations.

**Next → Lab 3.3:** We swap the symmetric GCN aggregation for **learnable, per-edge attention weights** (GAT). Can the model learn to focus on the most relevant neighbours — and can we visualise what it pays attention to?